# 面试问题：KV Cache 量化为什么 K 常按 channel、V 常按 token？怎样实现 2-bit cache？

**回答主线。** KV Cache 随层数、序列长度、KV head 和 head dimension 线性增长，并在 decode 每步被读取，因此同时占容量和带宽。Weight-only 量化不能解决这部分。K/V 的异常值分布不同：一种常见设计对 K 沿 token 维为每个 channel 定标，对 V 为每个 token 沿 channel 定标，并把最近窗口保留高精度以降低在线校准和注意力误差。

下面实现非对称低比特量化、轴选择、2-bit packing、残差窗口、分块在线追加与 attention 误差门禁。NumPy 代码验证数值与状态语义，不代表硬件上已有融合反量化 kernel。


In [ ]:
import hashlib, json, math  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

# 受控 K/V 分布用于比较不同归约轴，而非宣称真实模型统计。
rng154 = np.random.default_rng(154)  # 计算并保存当前步骤的中间状态。

def mse154(a, b):  # 定义本节可复用的核心函数。
    return float(np.mean((np.asarray(a, dtype=np.float64) - np.asarray(b, dtype=np.float64)) ** 2))  # 返回当前分支计算出的结果。

probe154 = rng154.normal(size=(8, 6))  # 计算并保存当前步骤的中间状态。
assert mse154(probe154, probe154) == 0.0  # 用受控断言验证关键不变量。
assert probe154.shape == (8, 6)  # 用受控断言验证关键不变量。
assert np.isfinite(probe154).all()  # 用受控断言验证关键不变量。


## 1. 先计算未量化 KV 容量

每层需要 Key 和 Value 两份，容量为 `2 * layers * batch * tokens * kv_heads * head_dim * bytes`。GQA 减少 `kv_heads`，量化减少 `bytes`，二者是可叠加但不同的优化。


In [ ]:
def kv_bytes154(layers, batch, tokens, kv_heads, head_dim, bytes_per_value):  # 定义本节可复用的核心函数。
    # 最前面的 2 分别代表 Key 与 Value。
    fields = [layers, batch, tokens, kv_heads, head_dim, bytes_per_value]  # 计算并保存当前步骤的中间状态。
    if any(float(v) <= 0 for v in fields):  # 按当前条件选择后续控制路径。
        raise ValueError("all dimensions must be positive")  # 遇到非法合同立即显式失败。
    return int(2 * layers * batch * tokens * kv_heads * head_dim * bytes_per_value)  # 返回当前分支计算出的结果。

fp16_bytes154 = kv_bytes154(32, 4, 4096, 8, 128, 2)  # 计算并保存当前步骤的中间状态。
int2_payload154 = kv_bytes154(32, 4, 4096, 8, 128, 0.25)  # 计算并保存当前步骤的中间状态。
assert fp16_bytes154 == 4 * 32 * 4096 * 8 * 128 * 4  # 用受控断言验证关键不变量。
assert int2_payload154 == fp16_bytes154 // 8  # 用受控断言验证关键不变量。
assert fp16_bytes154 > int2_payload154  # 用受控断言验证关键不变量。


## 2. 非对称量化保存 min 与 scale

对 `b` bit 使用 `0..2^b-1` 整数码。每个量化组保存最小值和 scale，反量化为 `q*scale+min`。常量组必须给安全 scale，否则除零会污染整块。


In [ ]:
def affine_quant154(x, bits, reduce_axis):  # 定义本节可复用的核心函数。
    # reduce_axis 表示共享一组 qparam 的统计轴，并保留维度用于广播。
    x = np.asarray(x, dtype=np.float64)  # 计算并保存当前步骤的中间状态。
    levels = 2 ** bits - 1  # 计算并保存当前步骤的中间状态。
    xmin = x.min(axis=reduce_axis, keepdims=True)  # 计算并保存当前步骤的中间状态。
    xmax = x.max(axis=reduce_axis, keepdims=True)  # 计算并保存当前步骤的中间状态。
    scale = np.where(xmax > xmin, (xmax - xmin) / levels, 1.0)  # 计算并保存当前步骤的中间状态。
    q = np.clip(np.round((x - xmin) / scale), 0, levels).astype(np.uint8)  # 计算并保存当前步骤的中间状态。
    return q, scale, xmin  # 返回当前分支计算出的结果。

def affine_dequant154(q, scale, xmin):  # 定义本节可复用的核心函数。
    return q.astype(np.float64) * scale + xmin  # 返回当前分支计算出的结果。

q154, scale154, min154 = affine_quant154(probe154, bits=2, reduce_axis=1)  # 计算并保存当前步骤的中间状态。
dq154 = affine_dequant154(q154, scale154, min154)  # 计算并保存当前步骤的中间状态。
assert q154.min() >= 0 and q154.max() <= 3  # 用受控断言验证关键不变量。
assert dq154.shape == probe154.shape  # 用受控断言验证关键不变量。
assert np.isfinite(dq154).all()  # 用受控断言验证关键不变量。


## 3. K/V 的最佳 qparam 轴由异常值结构决定

这里把 K 构造成“不同 channel 量级差异大”，把 V 构造成“不同 token 量级差异大”。全局 MSE 会被最大量级组支配，因此按每个逻辑组的能量归一化后再平均：K 比较各 channel 的 NMSE，V 比较各 token 的 NMSE。这是受控反例，不应替代目标模型上的校准统计。


In [ ]:
# 合成两个正交的异常值模式，用来验证 qparam 归约轴的影响。
tokens154, dim154 = 64, 8  # 计算并保存当前步骤的中间状态。
time154 = np.linspace(-1.0, 1.0, tokens154)[:, None]  # 计算并保存当前步骤的中间状态。
channel_scales154 = np.geomspace(0.01, 20.0, dim154)[None, :]  # 计算并保存当前步骤的中间状态。
k154 = time154 * channel_scales154 + 0.01 * rng154.normal(size=(tokens154, dim154))  # 计算并保存当前步骤的中间状态。
token_scales154 = np.geomspace(0.01, 20.0, tokens154)[:, None]  # 计算并保存当前步骤的中间状态。
pattern154 = np.linspace(-1.0, 1.0, dim154)[None, :]  # 计算并保存当前步骤的中间状态。
v154 = token_scales154 * pattern154 + 0.01 * rng154.normal(size=(tokens154, dim154))  # 计算并保存当前步骤的中间状态。

k_pc154 = affine_dequant154(*affine_quant154(k154, 2, reduce_axis=0)[0:])  # 计算并保存当前步骤的中间状态。
k_pt154 = affine_dequant154(*affine_quant154(k154, 2, reduce_axis=1)[0:])  # 计算并保存当前步骤的中间状态。
v_pc154 = affine_dequant154(*affine_quant154(v154, 2, reduce_axis=0)[0:])  # 计算并保存当前步骤的中间状态。
v_pt154 = affine_dequant154(*affine_quant154(v154, 2, reduce_axis=1)[0:])  # 计算并保存当前步骤的中间状态。

def grouped_nmse154(reference, candidate, reduce_axis):  # 定义本节可复用的核心函数。
    # 先在每个逻辑组内除以信号能量，再对组平均，避免大组垄断全局 MSE。
    error = np.mean((reference - candidate) ** 2, axis=reduce_axis)  # 计算并保存当前步骤的中间状态。
    energy = np.maximum(np.mean(reference ** 2, axis=reduce_axis), 1e-12)  # 计算并保存当前步骤的中间状态。
    return float(np.mean(error / energy))  # 返回当前分支计算出的结果。

assert grouped_nmse154(k154, k_pc154, 0) < grouped_nmse154(k154, k_pt154, 0)  # 用受控断言验证关键不变量。
assert grouped_nmse154(v154, v_pt154, 1) < grouped_nmse154(v154, v_pc154, 1)  # 用受控断言验证关键不变量。
assert k_pc154.shape == v_pt154.shape == (tokens154, dim154)  # 用受控断言验证关键不变量。


## 4. Residual window 把最近 token 保持高精度

新 token 每步到达，立刻重算所有历史 qparam 很昂贵。常见办法保留最近 `R` 个 token 为 FP16/BF16，窗口满后把较老的一组转入低比特块。这样既支持 append，也保护 attention 最敏感的近邻。


In [ ]:
def residual_quantize154(k, v, residual):  # 定义本节可复用的核心函数。
    # 历史 K 按列、历史 V 按行量化；最近窗口原样拼回。
    split = max(0, len(k) - residual)  # 计算并保存当前步骤的中间状态。
    if split == 0:  # 按当前条件选择后续控制路径。
        return k.copy(), v.copy(), {"quantized_tokens": 0}  # 返回当前分支计算出的结果。
    k_old = affine_dequant154(*affine_quant154(k[:split], 2, reduce_axis=0)[0:])  # 计算并保存当前步骤的中间状态。
    v_old = affine_dequant154(*affine_quant154(v[:split], 2, reduce_axis=1)[0:])  # 计算并保存当前步骤的中间状态。
    return np.vstack([k_old, k[split:]]), np.vstack([v_old, v[split:]]), {"quantized_tokens": split}  # 返回当前分支计算出的结果。

kr154, vr154, residual_meta154 = residual_quantize154(k154, v154, residual=8)  # 计算并保存当前步骤的中间状态。
assert residual_meta154["quantized_tokens"] == 56  # 用受控断言验证关键不变量。
assert np.array_equal(kr154[-8:], k154[-8:])  # 用受控断言验证关键不变量。
assert np.array_equal(vr154[-8:], v154[-8:])  # 用受控断言验证关键不变量。


## 5. 2-bit payload 要四个整数码打进一个 byte

只把 NumPy dtype 写成 `uint8` 仍是 8 bit。真正的 2-bit payload 需要位打包，并额外保存 qparam、shape、轴和尾部长度。下面实现四码一字节的可逆 oracle。


In [ ]:
def pack2bit154(codes):  # 定义本节可复用的核心函数。
    # 每四个码依次放入 byte 的 bit [1:0]、[3:2]、[5:4]、[7:6]。
    flat = np.asarray(codes, dtype=np.uint8).ravel()  # 计算并保存当前步骤的中间状态。
    if np.any(flat > 3):  # 按当前条件选择后续控制路径。
        raise ValueError("2-bit code out of range")  # 遇到非法合同立即显式失败。
    padded = np.pad(flat, (0, (-len(flat)) % 4))  # 计算并保存当前步骤的中间状态。
    groups = padded.reshape(-1, 4)  # 计算并保存当前步骤的中间状态。
    packed = groups[:, 0] | (groups[:, 1] << 2) | (groups[:, 2] << 4) | (groups[:, 3] << 6)  # 计算并保存当前步骤的中间状态。
    return packed.astype(np.uint8), len(flat)  # 返回当前分支计算出的结果。

def unpack2bit154(packed, length):  # 定义本节可复用的核心函数。
    packed = np.asarray(packed, dtype=np.uint8)  # 计算并保存当前步骤的中间状态。
    values = np.stack([(packed >> shift) & 3 for shift in (0, 2, 4, 6)], axis=1).ravel()  # 计算并保存当前步骤的中间状态。
    return values[:length]  # 返回当前分支计算出的结果。

packed154, length154 = pack2bit154(q154)  # 计算并保存当前步骤的中间状态。
unpacked154 = unpack2bit154(packed154, length154)  # 计算并保存当前步骤的中间状态。
assert np.array_equal(unpacked154, q154.ravel())  # 用受控断言验证关键不变量。
assert len(packed154) == math.ceil(q154.size / 4)  # 用受控断言验证关键不变量。
assert packed154.dtype == np.uint8  # 用受控断言验证关键不变量。


## 6. 在线追加按 token block 冻结 qparam

K 的 per-channel scale 需要多个 token 才能统计，因此可按固定 token block 校准并冻结；不同 block 不能误用彼此 qparam。块索引与绝对 token 范围属于 cache metadata。


In [ ]:
@dataclass  # 为下方定义附加声明式配置。
class QuantBlock154:  # 定义承载本节状态与行为的数据结构。
    start: int  # 执行当前语句以推进本节示例。
    stop: int  # 执行当前语句以推进本节示例。
    q: np.ndarray  # 执行当前语句以推进本节示例。
    scale: np.ndarray  # 执行当前语句以推进本节示例。
    xmin: np.ndarray  # 执行当前语句以推进本节示例。

def quantize_k_blocks154(k, block_tokens):  # 定义本节可复用的核心函数。
    # 每块独立按 channel 定标，尾块允许不足 block_tokens。
    blocks = []  # 计算并保存当前步骤的中间状态。
    for start in range(0, len(k), block_tokens):  # 遍历输入元素以累积或检查结果。
        stop = min(start + block_tokens, len(k))  # 计算并保存当前步骤的中间状态。
        q, scale, xmin = affine_quant154(k[start:stop], 2, reduce_axis=0)  # 计算并保存当前步骤的中间状态。
        blocks.append(QuantBlock154(start, stop, q, scale, xmin))  # 执行当前语句以推进本节示例。
    return blocks  # 返回当前分支计算出的结果。

blocks154 = quantize_k_blocks154(k154, block_tokens=16)  # 计算并保存当前步骤的中间状态。
rebuilt_k154 = np.vstack([affine_dequant154(b.q, b.scale, b.xmin) for b in blocks154])  # 计算并保存当前步骤的中间状态。
assert [(b.start, b.stop) for b in blocks154] == [(0, 16), (16, 32), (32, 48), (48, 64)]  # 用受控断言验证关键不变量。
assert rebuilt_k154.shape == k154.shape  # 用受控断言验证关键不变量。
assert all(b.q.max() <= 3 for b in blocks154)  # 用受控断言验证关键不变量。


## 7. 最终门禁看 attention 输出，不只看 K/V MSE

K 的误差会先改变 softmax 权重，V 的误差再改变加权和；二者对任务影响非线性。应在不同层、头、长度和数据 slice 上比较 attention/logit/任务指标。


In [ ]:
def attention_one154(query, k, v):  # 定义本节可复用的核心函数。
    # 单 query oracle 使用稳定 softmax，直接暴露 cache 误差到输出。
    score = query @ k.T / math.sqrt(k.shape[1])  # 计算并保存当前步骤的中间状态。
    weight = np.exp(score - np.max(score))  # 计算并保存当前步骤的中间状态。
    weight /= weight.sum()  # 计算并保存当前步骤的中间状态。
    return weight @ v, weight  # 返回当前分支计算出的结果。

query154 = rng154.normal(size=dim154)  # 计算并保存当前步骤的中间状态。
exact_out154, exact_w154 = attention_one154(query154, k154, v154)  # 计算并保存当前步骤的中间状态。
quant_out154, quant_w154 = attention_one154(query154, kr154, vr154)  # 计算并保存当前步骤的中间状态。
attention_rel154 = np.linalg.norm(exact_out154 - quant_out154) / max(np.linalg.norm(exact_out154), 1e-12)  # 计算并保存当前步骤的中间状态。
assert np.isclose(exact_w154.sum(), 1.0) and np.isclose(quant_w154.sum(), 1.0)  # 用受控断言验证关键不变量。
assert np.isfinite(attention_rel154)  # 用受控断言验证关键不变量。
assert attention_rel154 < 0.6  # 用受控断言验证关键不变量。


## 8. Cache 制品绑定模型、层、轴、block 与 residual 策略

量化 payload 若缺少 head 布局、qparam 轴或模型 revision，可能成功反量化却产生错误结果。发布还要测融合 kernel 的 TPOT、峰值 HBM、batch capacity 与端到端质量。


In [ ]:
def cache_manifest154(config, payloads):  # 定义本节可复用的核心函数。
    # 每个 payload 摘要与配置共同进入总摘要，避免元数据/数据错配。
    part_digests = [hashlib.sha256(p.tobytes()).hexdigest() for p in payloads]  # 计算并保存当前步骤的中间状态。
    canonical = json.dumps({"config": config, "parts": part_digests}, sort_keys=True, separators=(",", ":"))  # 计算并保存当前步骤的中间状态。
    return canonical, hashlib.sha256(canonical.encode()).hexdigest()  # 返回当前分支计算出的结果。

config154 = {"model": "toy-v1", "bits": 2, "k_axis": "channel", "v_axis": "token", "block": 16, "residual": 8}  # 计算并保存当前步骤的中间状态。
manifest154, digest154 = cache_manifest154(config154, [b.q for b in blocks154])  # 计算并保存当前步骤的中间状态。
assert len(digest154) == 64  # 用受控断言验证关键不变量。
assert json.loads(manifest154)["config"]["k_axis"] == "channel"  # 用受控断言验证关键不变量。
assert len(json.loads(manifest154)["parts"]) == 4  # 用受控断言验证关键不变量。


## 面试总结

- KV 容量与 `layers * tokens * kv_heads * head_dim * bytes` 成正比，且 decode 会反复读取。
- K per-channel、V per-token 是基于异常值结构的常见设计，应在具体模型校准，而不是背成永恒规则。
- 真正 2-bit 需要位打包；qparam、块索引、残差窗口与绝对位置也占空间并参与版本合同。
- 最终用 attention/logit/任务质量和真实 kernel TPOT 验收，K/V MSE 只负责定位。

延伸阅读：[KIVI](https://arxiv.org/abs/2402.02750)、[KVQuant](https://arxiv.org/abs/2401.18079)、[Hugging Face Quantized Cache](https://huggingface.co/docs/transformers/main/en/kv_cache#quantized-cache)。
